# House Prices: 연식·미래연도 이상 플래그 ablation

## tl;dr

원본 연도 reference의 5-fold RMSLE는 **0.150372 ± 0.040674**,
연식·이상 플래그 실험은 **0.152192 ± 0.039693**다.
CV 평균 변화는 **+0.001820**, OOF RMSLE 변화는 **+0.001568**이며
3/5 folds가 개선됐다. 현재 결정은 **reject**이다.
특정 행의 연도 보정과 외부 Python 스크립트 의존은 사용하지 않았다.

## Context & Methods

이 self-contained 노트북은 외부 프로젝트 스크립트를 import하지 않고 다음 두 조건을
동일한 BaseMLP 학습 설정에서 비교하는 실행형 실험 기록이다.

1. 원본 연도 값을 그대로 사용하는 reference
2. 판매 시점 연식 3개와 `FutureYearAnomaly`를 추가하는 feature 실험

### Key Assumptions

- `KFold(n_splits=5, shuffle=True, random_state=42)`를 두 조건에 동일하게 사용한다.
- 모든 1,460행은 학습 대상과 OOF validation에 유지한다.
- `Id`와 `SalePrice`는 모델 입력에서 제외한다.
- 피처 생성에는 타깃이나 개별 `Id`를 사용하지 않는다.
- 음수 연식은 오류 가능성이 있는 값으로 보고 결측 처리한 뒤, fold 학습 부분의 중앙값으로 대체한다.
- `nn.Module`, 전처리, 직접 구현한 학습·검증 루프, optimizer, loss, 초기화,
  early stopping과 실험 로깅을 이 노트북 안에 모두 정의한다.

## Data

입력은 `data/train.csv`이며 `keep_default_na=False`로 읽어 구조적 부재를 나타내는
`NA`/`None` 문자열을 먼저 보존한다. 이 ablation은 test predictor를 읽거나
leaderboard 결과를 사용하지 않는다.

In [1]:
from __future__ import annotations

import csv
import hashlib
import json
import random
import time
from datetime import datetime, timezone
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
import torch
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists():
    ROOT = ROOT.parent.parent

TRAIN_PATH = ROOT / "data" / "train.csv"
EXPERIMENTS_PATH = ROOT / "reports" / "experiments.csv"
NOTEBOOK_PATH = (
    ROOT / "notebooks" / "feature_engineering" /
    "house_prices_age_anomaly_ablation.ipynb"
)
SUMMARY_PATH = ROOT / "reports" / "basemlp_age_anomaly_notebook_summary.json"
REFERENCE_ID = "BASEMLP-20260719-RAWYEAR-REF-NB-03"
FEATURE_ID = "BASEMLP-20260719-FEAT-AGE-ANOMALY-NB-03"

SEED = 42
N_SPLITS = 5
HIDDEN_DIMS = (128, 64)
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
BATCH_SIZE = 64
MAX_EPOCHS = 200
PATIENCE = 25
MIN_DELTA = 1e-5
DEVICE = torch.device("cpu")

NUMERIC_COLUMNS = [
    "LotFrontage", "LotArea", "OverallQual", "OverallCond", "YearBuilt",
    "YearRemodAdd", "MasVnrArea", "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF",
    "TotalBsmtSF", "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea",
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath", "BedroomAbvGr",
    "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces", "GarageYrBlt", "GarageCars",
    "GarageArea", "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch",
    "ScreenPorch", "PoolArea", "MiscVal", "YrSold",
]
BASEMENT_COLUMNS = [
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2"
]
GARAGE_COLUMNS = ["GarageType", "GarageFinish", "GarageQual", "GarageCond"]
AGE_FEATURE_COLUMNS = [
    "HouseAgeAtSale", "RemodAgeAtSale", "GarageAgeAtSale", "FutureYearAnomaly"
]


class BaseMLP(nn.Module):
    def __init__(self, input_dim: int) -> None:
        super().__init__()
        self.hidden1 = nn.Linear(input_dim, HIDDEN_DIMS[0])
        self.hidden2 = nn.Linear(HIDDEN_DIMS[0], HIDDEN_DIMS[1])
        self.output = nn.Linear(HIDDEN_DIMS[1], 1)
        self.activation = nn.ReLU()
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_normal_(self.hidden1.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden1.bias)
        nn.init.kaiming_normal_(self.hidden2.weight, nonlinearity="relu")
        nn.init.zeros_(self.hidden2.bias)
        nn.init.xavier_normal_(self.output.weight)
        nn.init.zeros_(self.output.bias)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        hidden = self.activation(self.hidden1(inputs))
        hidden = self.activation(self.hidden2(hidden))
        return self.output(hidden).squeeze(1)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def predict_log_target(
    model: BaseMLP,
    features: np.ndarray,
    target_mean: float,
    target_std: float,
) -> np.ndarray:
    model.eval()
    with torch.no_grad():
        tensor = torch.as_tensor(features, dtype=torch.float32, device=DEVICE)
        standardized = model(tensor).cpu().numpy().astype(np.float64)
    return standardized * target_std + target_mean


def train_fold(
    X_train: np.ndarray,
    y_train_log: np.ndarray,
    X_valid: np.ndarray,
    y_valid_log: np.ndarray,
    seed: int,
    experiment_id: str,
    fold: int,
    checkpoint_path: Path,
    epoch_log_path: Path,
) -> tuple[BaseMLP, dict[str, float | int]]:
    set_seed(seed)
    target_mean = float(y_train_log.mean())
    target_std = float(y_train_log.std(ddof=0))
    y_train_standardized = (y_train_log - target_mean) / target_std
    y_valid_standardized = (y_valid_log - target_mean) / target_std

    train_dataset = TensorDataset(
        torch.as_tensor(X_train, dtype=torch.float32),
        torch.as_tensor(y_train_standardized, dtype=torch.float32),
    )
    generator = torch.Generator().manual_seed(seed)
    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        generator=generator,
        num_workers=0,
    )
    X_valid_tensor = torch.as_tensor(X_valid, dtype=torch.float32, device=DEVICE)
    y_valid_tensor = torch.as_tensor(
        y_valid_standardized, dtype=torch.float32, device=DEVICE
    )

    model = BaseMLP(X_train.shape[1]).to(DEVICE)
    loss_fn = nn.MSELoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    best_score = float("inf")
    best_epoch = 0
    epochs_without_improvement = 0
    epoch_rows: list[dict[str, float | int]] = []

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        train_loss_sum = 0.0
        train_count = 0
        for batch_features, batch_target in train_loader:
            batch_features = batch_features.to(DEVICE)
            batch_target = batch_target.to(DEVICE)
            optimizer.zero_grad()
            prediction = model(batch_features)
            loss = loss_fn(prediction, batch_target)
            loss.backward()
            optimizer.step()
            train_loss_sum += float(loss.detach().cpu()) * len(batch_features)
            train_count += len(batch_features)

        model.eval()
        with torch.no_grad():
            valid_standardized = model(X_valid_tensor)
            valid_loss = loss_fn(valid_standardized, y_valid_tensor)
            valid_log_prediction = (
                valid_standardized.cpu().numpy().astype(np.float64) * target_std
                + target_mean
            )
        valid_rmsle = float(
            np.sqrt(np.mean((y_valid_log - valid_log_prediction) ** 2))
        )
        epoch_rows.append(
            {
                "epoch": epoch,
                "train_loss": train_loss_sum / train_count,
                "validation_loss": float(valid_loss.cpu()),
                "validation_rmsle": valid_rmsle,
                "learning_rate": float(optimizer.param_groups[0]["lr"]),
            }
        )

        if valid_rmsle < best_score - MIN_DELTA:
            best_score = valid_rmsle
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(
                {
                    "experiment_id": experiment_id,
                    "fold": fold,
                    "input_dim": int(X_train.shape[1]),
                    "model_state_dict": model.state_dict(),
                    "target_mean": target_mean,
                    "target_std": target_std,
                    "architecture": [*HIDDEN_DIMS, 1],
                    "seed": seed,
                },
                checkpoint_path,
            )
        else:
            epochs_without_improvement += 1
        if epochs_without_improvement >= PATIENCE:
            break

    pd.DataFrame(epoch_rows).to_csv(epoch_log_path, index=False)
    checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    restored_prediction = predict_log_target(
        model, X_valid, checkpoint["target_mean"], checkpoint["target_std"]
    )
    restored_rmsle = float(
        np.sqrt(np.mean((y_valid_log - restored_prediction) ** 2))
    )
    if not np.isclose(restored_rmsle, best_score, rtol=0, atol=1e-6):
        raise RuntimeError("Restored checkpoint score does not match the saved best score.")
    return model, {
        "best_epoch": best_epoch,
        "stopping_epoch": epoch,
        "best_validation_rmsle": best_score,
        "restored_validation_rmsle": restored_rmsle,
        "target_mean": target_mean,
        "target_std": target_std,
    }


def append_experiment(row: dict[str, str]) -> None:
    with EXPERIMENTS_PATH.open(newline="", encoding="utf-8") as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        existing_ids = {existing["experiment_id"] for existing in reader}
    if fieldnames is None:
        raise RuntimeError("reports/experiments.csv has no header.")
    if row["experiment_id"] in existing_ids:
        raise RuntimeError(f"Experiment ID already exists: {row['experiment_id']}")
    with EXPERIMENTS_PATH.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writerow({field: row.get(field, "") for field in fieldnames})


train = pd.read_csv(TRAIN_PATH, keep_default_na=False)
assert train.shape == (1460, 81)
assert train["Id"].is_unique
assert train["SalePrice"].gt(0).all()
categorical_columns = [
    column
    for column in train.columns
    if column not in {*NUMERIC_COLUMNS, "Id", "SalePrice"}
]
assert len(NUMERIC_COLUMNS) == 34
assert len(categorical_columns) == 45

y = train["SalePrice"].to_numpy(dtype=np.float64)
y_log = np.log1p(y)
print(f"train: {train.shape[0]:,} rows × {train.shape[1]} columns")
print(f"train sha256: {sha256(TRAIN_PATH)}")
print("external project script imports: 0")

train: 1,460 rows × 81 columns
train sha256: 1e18addf81e5e4d347cc17ee6075bbe4a42b7fa26b9e5b063e8f692a5f929d41
external project script imports: 0


### Prepare raw-year reference and age features

구조적 부재 처리는 기존 BaseMLP와 같지만 특정 행의 연도는 수정하지 않는다.
`FutureYearAnomaly`는 세 연식 중 하나라도 음수인 경우에만 1이다. 차고가 없어
`GarageAgeAtSale`이 결측인 경우는 미래연도 이상으로 계산하지 않는다.

In [2]:
def clean_rows_without_record_correction(
    frame: pd.DataFrame,
    categorical_columns: list[str],
) -> pd.DataFrame:
    cleaned = frame.copy()
    for column in NUMERIC_COLUMNS:
        cleaned[column] = pd.to_numeric(
            cleaned[column].replace({"NA": np.nan, "": np.nan}),
            errors="coerce",
        )

    basement_absent = cleaned["TotalBsmtSF"].fillna(0).eq(0)
    garage_absent = (
        cleaned["GarageCars"].fillna(0).eq(0)
        & cleaned["GarageArea"].fillna(0).eq(0)
    )

    for column in BASEMENT_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & basement_absent, column] = "NoBasement"
        cleaned.loc[missing & ~basement_absent, column] = "Unknown"

    for column in GARAGE_COLUMNS:
        missing = cleaned[column].isin(["NA", ""])
        cleaned.loc[missing & garage_absent, column] = "NoGarage"
        cleaned.loc[missing & ~garage_absent, column] = "Unknown"
    cleaned.loc[garage_absent, "GarageYrBlt"] = 0.0

    fireplace_absent = cleaned["Fireplaces"].fillna(0).eq(0)
    pool_absent = cleaned["PoolArea"].fillna(0).eq(0)
    cleaned.loc[
        cleaned["FireplaceQu"].isin(["NA", ""]) & fireplace_absent,
        "FireplaceQu",
    ] = "NoFireplace"
    cleaned.loc[
        cleaned["PoolQC"].isin(["NA", ""]) & pool_absent,
        "PoolQC",
    ] = "NoPool"

    for column, label in {
        "Alley": "NoAlley",
        "Fence": "NoFence",
        "MiscFeature": "NoMiscFeature",
    }.items():
        cleaned[column] = cleaned[column].replace({"NA": label, "": label})

    cleaned["MSSubClass"] = cleaned["MSSubClass"].map(
        lambda value: f"SC{value}"
    )
    cleaned["MoSold"] = cleaned["MoSold"].map(
        lambda value: f"M{int(value):02d}"
    )
    for column in categorical_columns:
        cleaned[column] = cleaned[column].replace(
            {"NA": "Unknown", "": "Unknown"}
        )
    return cleaned


def add_age_features(frame: pd.DataFrame) -> pd.DataFrame:
    featured = frame.copy()
    sale_year = featured["YrSold"]
    garage_present = (
        featured["GarageCars"].fillna(0).gt(0)
        | featured["GarageArea"].fillna(0).gt(0)
    )
    ages = {
        "HouseAgeAtSale": sale_year - featured["YearBuilt"],
        "RemodAgeAtSale": sale_year - featured["YearRemodAdd"],
        "GarageAgeAtSale": (
            sale_year - featured["GarageYrBlt"]
        ).where(garage_present),
    }
    future_year = pd.Series(False, index=featured.index)
    for column, age in ages.items():
        invalid = age.lt(0)
        future_year |= invalid.fillna(False)
        featured[column] = age.mask(invalid)
    featured["FutureYearAnomaly"] = future_year.astype(float)
    return featured


raw_year_X = clean_rows_without_record_correction(
    train.drop(columns="SalePrice"), categorical_columns
)
age_feature_X = add_age_features(raw_year_X)

anomaly_rows = int(age_feature_X["FutureYearAnomaly"].sum())
negative_age_counts = {
    column: int(age_feature_X[column].dropna().lt(0).sum())
    for column in AGE_FEATURE_COLUMNS[:-1]
}
assert anomaly_rows == 1
assert all(count == 0 for count in negative_age_counts.values())
assert raw_year_X["YearRemodAdd"].equals(
    pd.to_numeric(
        train["YearRemodAdd"].replace({"NA": np.nan, "": np.nan}),
        errors="coerce",
    )
)

display(
    pd.DataFrame(
        {
            "check": [
                "row-specific year corrections",
                "future-year anomaly rows",
                "negative generated ages after handling",
            ],
            "value": [0, anomaly_rows, sum(negative_age_counts.values())],
        }
    )
)

,check,value
0,row-specific year corrections,0
1,future-year anomaly rows,1
2,negative generated ages after handling,0


## Results

각 fold에서 전처리기와 모델을 새로 적합하고 validation RMSLE 기준 best checkpoint를
저장·복원한다. reference와 feature 조건은 같은 fold index와 fold seed를 사용한다.

In [3]:
def make_preprocessor(numeric_columns: list[str]) -> ColumnTransformer:
    numeric = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )
    categorical = Pipeline(
        [
            (
                "imputer",
                SimpleImputer(strategy="constant", fill_value="Unknown"),
            ),
            (
                "onehot",
                OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ),
        ]
    )
    return ColumnTransformer(
        [
            ("numeric", numeric, numeric_columns),
            ("categorical", categorical, categorical_columns),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )


def run_experiment(
    experiment_id: str,
    X: pd.DataFrame,
    numeric_columns: list[str],
    generated_features: list[str],
) -> dict[str, object]:
    started = time.perf_counter()
    checkpoint_dir = ROOT / "artifacts" / "checkpoints" / experiment_id
    preprocessor_dir = ROOT / "artifacts" / "preprocessors" / experiment_id
    epoch_log_dir = ROOT / "artifacts" / "logs" / experiment_id
    for directory in (checkpoint_dir, preprocessor_dir, epoch_log_dir):
        directory.mkdir(parents=True, exist_ok=False)

    metrics_path = ROOT / "reports" / f"basemlp_metrics_{experiment_id}.json"
    oof_path = ROOT / "reports" / f"basemlp_oof_{experiment_id}.csv"
    folds = list(
        KFold(
            n_splits=N_SPLITS,
            shuffle=True,
            random_state=SEED,
        ).split(X)
    )
    oof_log_prediction = np.full(len(X), np.nan, dtype=np.float64)
    fold_rows: list[dict[str, object]] = []

    for fold, (train_index, valid_index) in enumerate(folds, start=1):
        fold_seed = SEED + fold
        checkpoint_path = checkpoint_dir / f"fold_{fold}_best.pt"
        preprocessor_path = (
            preprocessor_dir / f"fold_{fold}_preprocessor.joblib"
        )
        epoch_log_path = epoch_log_dir / f"fold_{fold}_epochs.csv"

        preprocessor = make_preprocessor(numeric_columns)
        X_train = preprocessor.fit_transform(
            X.iloc[train_index]
        ).astype(np.float32)
        X_valid = preprocessor.transform(
            X.iloc[valid_index]
        ).astype(np.float32)
        joblib.dump(preprocessor, preprocessor_path)

        _, training = train_fold(
            X_train,
            y_log[train_index],
            X_valid,
            y_log[valid_index],
            seed=fold_seed,
            experiment_id=experiment_id,
            fold=fold,
            checkpoint_path=checkpoint_path,
            epoch_log_path=epoch_log_path,
        )

        saved_preprocessor = joblib.load(preprocessor_path)
        X_valid_saved = saved_preprocessor.transform(
            X.iloc[valid_index]
        ).astype(np.float32)
        checkpoint = torch.load(
            checkpoint_path,
            map_location=DEVICE,
            weights_only=False,
        )
        restored_model = BaseMLP(checkpoint["input_dim"]).to(DEVICE)
        restored_model.load_state_dict(checkpoint["model_state_dict"])
        valid_prediction = predict_log_target(
            restored_model,
            X_valid_saved,
            checkpoint["target_mean"],
            checkpoint["target_std"],
        )
        oof_log_prediction[valid_index] = valid_prediction
        fold_rmsle = float(
            np.sqrt(
                np.mean(
                    (y_log[valid_index] - valid_prediction) ** 2
                )
            )
        )
        fold_rows.append(
            {
                "fold": fold,
                "train_rows": int(len(train_index)),
                "validation_rows": int(len(valid_index)),
                "encoded_features": int(X_train.shape[1]),
                "seed": fold_seed,
                "best_epoch": int(training["best_epoch"]),
                "stopping_epoch": int(training["stopping_epoch"]),
                "rmsle": fold_rmsle,
            }
        )

    assert not np.isnan(oof_log_prediction).any()
    fold_scores = np.array(
        [row["rmsle"] for row in fold_rows], dtype=np.float64
    )
    cv_mean = float(fold_scores.mean())
    cv_std = float(fold_scores.std(ddof=1))
    oof_rmsle = float(
        np.sqrt(np.mean((y_log - oof_log_prediction) ** 2))
    )
    duration_seconds = time.perf_counter() - started

    pd.DataFrame(
        {
            "Id": train["Id"],
            "SalePrice": y,
            "actual_log1p": y_log,
            "oof_log_prediction": oof_log_prediction,
            "oof_saleprice_prediction": np.clip(
                np.expm1(oof_log_prediction), 0, None
            ),
        }
    ).to_csv(oof_path, index=False)

    metrics = {
        "run_timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "experiment_id": experiment_id,
        "source": {
            "train_path": "data/train.csv",
            "train_sha256": sha256(TRAIN_PATH),
            "rows": int(len(train)),
        },
        "features": {
            "generated": generated_features,
            "row_specific_corrections": 0,
            "future_year_anomaly_rows": anomaly_rows,
            "negative_generated_ages_after_handling": negative_age_counts,
        },
        "validation": {
            "splitter": "KFold",
            "n_splits": N_SPLITS,
            "shuffle": True,
            "random_state": SEED,
            "metric": "RMSLE / RMSE on log1p target",
            "all_rows_retained": True,
        },
        "model": {
            "class": "BaseMLP(nn.Module)",
            "architecture": f"input->{list(HIDDEN_DIMS)}->1",
            "optimizer": "Adam",
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "loss": "MSELoss on fold-standardized log1p(SalePrice)",
            "batch_size": BATCH_SIZE,
            "max_epochs": MAX_EPOCHS,
            "patience": PATIENCE,
            "min_delta": MIN_DELTA,
            "device": str(DEVICE),
            "precision": "float32",
        },
        "results": {
            "fold_scores": fold_scores.tolist(),
            "cv_mean": cv_mean,
            "cv_std": cv_std,
            "oof_rmsle": oof_rmsle,
            "duration_seconds": duration_seconds,
        },
        "folds": fold_rows,
        "artifacts": {
            "notebook": str(NOTEBOOK_PATH.relative_to(ROOT)),
            "metrics": str(metrics_path.relative_to(ROOT)),
            "oof_predictions": str(oof_path.relative_to(ROOT)),
            "checkpoint_dir": str(checkpoint_dir.relative_to(ROOT)),
            "preprocessor_dir": str(preprocessor_dir.relative_to(ROOT)),
            "epoch_log_dir": str(epoch_log_dir.relative_to(ROOT)),
        },
        "kaggle_scores": {"public": "unverified", "private": "unverified"},
    }
    metrics_path.write_text(
        json.dumps(metrics, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    return metrics


reference_metrics = run_experiment(
    REFERENCE_ID,
    raw_year_X,
    list(NUMERIC_COLUMNS),
    [],
)
feature_metrics = run_experiment(
    FEATURE_ID,
    age_feature_X,
    [*NUMERIC_COLUMNS, *AGE_FEATURE_COLUMNS],
    AGE_FEATURE_COLUMNS,
)

In [4]:
reference_results = reference_metrics["results"]
feature_results = feature_metrics["results"]
cv_delta = feature_results["cv_mean"] - reference_results["cv_mean"]
oof_delta = feature_results["oof_rmsle"] - reference_results["oof_rmsle"]
fold_deltas = np.array(feature_results["fold_scores"]) - np.array(
    reference_results["fold_scores"]
)
improved_folds = int((fold_deltas < 0).sum())
decision = "reject" if cv_delta >= 0 else "provisional_keep"

comparison = pd.DataFrame(
    [
        {
            "experiment_id": REFERENCE_ID,
            "condition": "raw-year reference",
            "cv_mean": reference_results["cv_mean"],
            "cv_std": reference_results["cv_std"],
            "oof_rmsle": reference_results["oof_rmsle"],
        },
        {
            "experiment_id": FEATURE_ID,
            "condition": "age + future-year anomaly",
            "cv_mean": feature_results["cv_mean"],
            "cv_std": feature_results["cv_std"],
            "oof_rmsle": feature_results["oof_rmsle"],
        },
    ]
)
fold_comparison = pd.DataFrame(
    {
        "fold": range(1, N_SPLITS + 1),
        "reference_rmsle": reference_results["fold_scores"],
        "feature_rmsle": feature_results["fold_scores"],
        "feature_minus_reference": fold_deltas,
    }
)
display(comparison.round(6))
display(fold_comparison.round(6))

run_timestamp = datetime.now(timezone.utc).isoformat()
common_hyperparameters = {
    "hidden_dims": list(HIDDEN_DIMS),
    "activation": "ReLU",
    "dropout": 0.0,
    "batch_normalization": False,
    "weight_decay": WEIGHT_DECAY,
    "target_standardization": "fold-train",
    "fold_seeds": list(range(SEED + 1, SEED + N_SPLITS + 1)),
    "all_rows_retained": True,
    "source_notebook": str(NOTEBOOK_PATH.relative_to(ROOT)),
}
append_experiment(
    {
        "experiment_id": REFERENCE_ID,
        "datetime": run_timestamp,
        "objective": "Create a raw-year BaseMLP reference in a separate feature-engineering notebook",
        "data_version": f"train sha256={sha256(TRAIN_PATH)}",
        "split_strategy": "KFold(n_splits=5,shuffle=True,random_state=42)",
        "folds": str(N_SPLITS),
        "seed": str(SEED),
        "metric": "RMSLE / RMSE on log1p target",
        "preprocessing": "No row-specific year correction; row-only structural labels; fold median/scaling/one-hot; fold-train target standardization",
        "features": "79 original predictors; original year values; no generated features; Id excluded",
        "model": "BaseMLP",
        "architecture": "input->[128,64]->1; ReLU; no dropout; no batch normalization",
        "optimizer": "Adam",
        "loss": "MSELoss on fold-standardized log1p(SalePrice)",
        "learning_rate": str(LEARNING_RATE),
        "batch_size": str(BATCH_SIZE),
        "max_epochs": str(MAX_EPOCHS),
        "patience": str(PATIENCE),
        "best_epoch": "folds=" + "|".join(
            str(row["best_epoch"]) for row in reference_metrics["folds"]
        ),
        "hyperparameters": json.dumps(common_hyperparameters, ensure_ascii=False),
        "cv_mean": str(reference_results["cv_mean"]),
        "cv_std": str(reference_results["cv_std"]),
        "checkpoint_path": reference_metrics["artifacts"]["checkpoint_dir"],
        "artifact_path": "; ".join(
            [
                str(NOTEBOOK_PATH.relative_to(ROOT)),
                reference_metrics["artifacts"]["metrics"],
                reference_metrics["artifacts"]["oof_predictions"],
            ]
        ),
        "result": "completed_reference_notebook",
        "interpretation": "Raw-year reference completed in the self-contained notebook.",
        "next_action": "Use only as the paired reference for the age-anomaly notebook ablation.",
    }
)
feature_hyperparameters = {
    **common_hyperparameters,
    "reference_experiment": REFERENCE_ID,
    "generated_features": AGE_FEATURE_COLUMNS,
    "negative_age_handling": "missing then fold-train median imputation",
    "future_year_anomaly_rows": anomaly_rows,
}
append_experiment(
    {
        "experiment_id": FEATURE_ID,
        "datetime": run_timestamp,
        "objective": "Evaluate sale-time ages with invalid-age missingness and a future-year anomaly flag",
        "data_version": f"train sha256={sha256(TRAIN_PATH)}",
        "split_strategy": "KFold(n_splits=5,shuffle=True,random_state=42)",
        "folds": str(N_SPLITS),
        "seed": str(SEED),
        "metric": "RMSLE / RMSE on log1p target",
        "preprocessing": "No row-specific year correction; negative ages to missing; row-only structural labels; fold median/scaling/one-hot; fold-train target standardization",
        "features": "79 original predictors + HouseAgeAtSale, RemodAgeAtSale, GarageAgeAtSale, FutureYearAnomaly; Id excluded",
        "model": "BaseMLP",
        "architecture": "input->[128,64]->1; ReLU; no dropout; no batch normalization",
        "optimizer": "Adam",
        "loss": "MSELoss on fold-standardized log1p(SalePrice)",
        "learning_rate": str(LEARNING_RATE),
        "batch_size": str(BATCH_SIZE),
        "max_epochs": str(MAX_EPOCHS),
        "patience": str(PATIENCE),
        "best_epoch": "folds=" + "|".join(
            str(row["best_epoch"]) for row in feature_metrics["folds"]
        ),
        "hyperparameters": json.dumps(feature_hyperparameters, ensure_ascii=False),
        "cv_mean": str(feature_results["cv_mean"]),
        "cv_std": str(feature_results["cv_std"]),
        "checkpoint_path": feature_metrics["artifacts"]["checkpoint_dir"],
        "artifact_path": "; ".join(
            [
                str(NOTEBOOK_PATH.relative_to(ROOT)),
                feature_metrics["artifacts"]["metrics"],
                feature_metrics["artifacts"]["oof_predictions"],
            ]
        ),
        "result": decision,
        "interpretation": (
            f"Paired notebook comparison: CV delta={cv_delta:+.6f}; "
            f"OOF delta={oof_delta:+.6f}; improved folds={improved_folds}/5."
        ),
        "next_action": (
            "Keep the feature logic isolated in this notebook and move to the next feature group."
            if decision == "reject"
            else "Confirm the directional result across additional split seeds."
        ),
    }
)

summary = {
    "notebook": str(NOTEBOOK_PATH.relative_to(ROOT)),
    "source": {
        "train_path": "data/train.csv",
        "train_sha256": sha256(TRAIN_PATH),
        "train_rows": int(len(train)),
    },
    "reference_experiment": REFERENCE_ID,
    "feature_experiment": FEATURE_ID,
    "reference": reference_results,
    "feature": feature_results,
    "comparison": {
        "cv_delta": float(cv_delta),
        "oof_delta": float(oof_delta),
        "fold_deltas": fold_deltas.tolist(),
        "improved_folds": improved_folds,
        "decision": decision,
    },
    "feature_audit": {
        "row_specific_corrections": 0,
        "future_year_anomaly_rows": anomaly_rows,
        "negative_generated_ages_after_handling": negative_age_counts,
    },
    "validation": {
        "notebook_executed_top_to_bottom": True,
        "external_script_dependency": False,
        "kaggle_scores": "unverified",
    },
}
SUMMARY_PATH.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(
    f"decision={decision}; CV delta={cv_delta:+.6f}; "
    f"OOF delta={oof_delta:+.6f}; improved folds={improved_folds}/5"
)

,experiment_id,condition,cv_mean,cv_std,oof_rmsle
0,BASEMLP-20260719-RAWYEAR-REF-NB-03,raw-year reference,0.150372,0.040674,0.154710
1,BASEMLP-20260719-FEAT-AGE-ANOMALY-NB-03,age + future-year anomaly,0.152192,0.039693,0.156278


,fold,reference_rmsle,feature_rmsle,feature_minus_reference
0,1,0.141188,0.143748,0.002560
1,2,0.131368,0.144179,0.012811
2,3,0.222249,0.220852,-0.001396
3,4,0.133601,0.132793,-0.000808
4,5,0.123455,0.119388,-0.004067


decision=reject; CV delta=+0.001820; OOF delta=+0.001568; improved folds=3/5


In [5]:
TEST_PATH = ROOT / "data" / "test.csv"
SAMPLE_SUBMISSION_PATH = ROOT / "data" / "sample_submission.csv"
BASELINE_MLP_SUBMISSION_PATH = ROOT / "submissions" / "submission_BASEMLP-20260719-2H-01.csv"
SUBMISSION_ID = f"{FEATURE_ID}-SUB-01"
SUBMISSION_PATH = ROOT / "submissions" / f"submission_{SUBMISSION_ID}.csv"
COMPARISON_PATH = ROOT / "reports" / "basemlp_age_anomaly_submission_comparison.json"

test = pd.read_csv(TEST_PATH, keep_default_na=False)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
baseline_submission = pd.read_csv(BASELINE_MLP_SUBMISSION_PATH)
assert test.shape == (1459, 80)
assert list(test.columns) == [column for column in train.columns if column != "SalePrice"]
assert list(sample_submission.columns) == ["Id", "SalePrice"]
assert sample_submission["Id"].equals(test["Id"])
assert baseline_submission["Id"].equals(test["Id"])

raw_year_test = clean_rows_without_record_correction(test, categorical_columns)
age_feature_test = add_age_features(raw_year_test)
feature_checkpoint_dir = ROOT / feature_metrics["artifacts"]["checkpoint_dir"]
feature_preprocessor_dir = ROOT / feature_metrics["artifacts"]["preprocessor_dir"]
fold_test_log_predictions = []
for fold in range(1, N_SPLITS + 1):
    preprocessor = joblib.load(
        feature_preprocessor_dir / f"fold_{fold}_preprocessor.joblib"
    )
    checkpoint = torch.load(
        feature_checkpoint_dir / f"fold_{fold}_best.pt",
        map_location=DEVICE,
        weights_only=False,
    )
    X_test = preprocessor.transform(age_feature_test).astype(np.float32)
    model = BaseMLP(checkpoint["input_dim"]).to(DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    fold_test_log_predictions.append(
        predict_log_target(
            model,
            X_test,
            checkpoint["target_mean"],
            checkpoint["target_std"],
        )
    )

mean_test_log_prediction = np.vstack(fold_test_log_predictions).mean(axis=0)
feature_submission = sample_submission.copy()
feature_submission["SalePrice"] = np.clip(
    np.expm1(mean_test_log_prediction), 0, None
)
SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
feature_submission.to_csv(SUBMISSION_PATH, index=False)

checks = {
    "columns": feature_submission.columns.tolist() == ["Id", "SalePrice"],
    "rows": len(feature_submission) == len(test),
    "id_order": feature_submission["Id"].equals(test["Id"]),
    "unique_ids": bool(feature_submission["Id"].is_unique),
    "finite_predictions": bool(np.isfinite(feature_submission["SalePrice"]).all()),
    "positive_predictions": bool(feature_submission["SalePrice"].gt(0).all()),
}
assert all(checks.values())

prediction_difference = (
    feature_submission["SalePrice"].to_numpy(dtype=np.float64)
    - baseline_submission["SalePrice"].to_numpy(dtype=np.float64)
)
comparison = {
    "submission_experiment_id": SUBMISSION_ID,
    "feature_experiment_id": FEATURE_ID,
    "feature_submission_path": str(SUBMISSION_PATH.relative_to(ROOT)),
    "baseline_submission_path": str(BASELINE_MLP_SUBMISSION_PATH.relative_to(ROOT)),
    "columns_equal": feature_submission.columns.tolist() == baseline_submission.columns.tolist(),
    "id_exact_equal": feature_submission["Id"].equals(baseline_submission["Id"]),
    "saleprice_exact_equal": bool(np.array_equal(
        feature_submission["SalePrice"].to_numpy(),
        baseline_submission["SalePrice"].to_numpy(),
    )),
    "different_prediction_rows": int(np.count_nonzero(prediction_difference)),
    "max_abs_saleprice_difference": float(np.max(np.abs(prediction_difference))),
    "submission_checks": checks,
    "kaggle_public_score": "unverified",
    "kaggle_private_score": "unverified",
}
COMPARISON_PATH.write_text(
    json.dumps(comparison, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

append_experiment(
    {
        "experiment_id": SUBMISSION_ID,
        "datetime": datetime.now(timezone.utc).isoformat(),
        "objective": "Create and compare the age-anomaly BaseMLP submission",
        "data_version": f"train sha256={sha256(TRAIN_PATH)}; test sha256={sha256(TEST_PATH)}",
        "split_strategy": "KFold(n_splits=5,shuffle=True,random_state=42)",
        "folds": str(N_SPLITS),
        "seed": str(SEED),
        "metric": "RMSLE / RMSE on log1p target",
        "preprocessing": "Self-contained notebook; no row-specific year correction; negative ages to missing; fold median/scaling/one-hot; fold-train target standardization",
        "features": "79 original predictors + HouseAgeAtSale, RemodAgeAtSale, GarageAgeAtSale, FutureYearAnomaly; Id excluded",
        "model": "BaseMLP",
        "architecture": "input->[128,64]->1; ReLU; no dropout; no batch normalization",
        "optimizer": "Adam",
        "loss": "MSELoss on fold-standardized log1p(SalePrice)",
        "learning_rate": str(LEARNING_RATE),
        "batch_size": str(BATCH_SIZE),
        "max_epochs": str(MAX_EPOCHS),
        "patience": str(PATIENCE),
        "best_epoch": "folds=" + "|".join(
            str(row["best_epoch"]) for row in feature_metrics["folds"]
        ),
        "hyperparameters": json.dumps(
            {
                "source_experiment": FEATURE_ID,
                "test_prediction_strategy": "mean of five restored-checkpoint log predictions then expm1",
                "external_script_dependency": False,
            },
            ensure_ascii=False,
        ),
        "cv_mean": str(feature_results["cv_mean"]),
        "cv_std": str(feature_results["cv_std"]),
        "checkpoint_path": feature_metrics["artifacts"]["checkpoint_dir"],
        "artifact_path": "; ".join(
            [
                str(NOTEBOOK_PATH.relative_to(ROOT)),
                str(SUBMISSION_PATH.relative_to(ROOT)),
                str(COMPARISON_PATH.relative_to(ROOT)),
                feature_metrics["artifacts"]["metrics"],
                feature_metrics["artifacts"]["oof_predictions"],
            ]
        ),
        "result": "submission_candidate_local_worse",
        "interpretation": (
            f"Exact prediction equality with baseline MLP={comparison['saleprice_exact_equal']}; "
            f"different rows={comparison['different_prediction_rows']}."
        ),
        "next_action": "Record a Kaggle score only after this exact CSV is submitted.",
    }
)

summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))
summary["submission"] = comparison
SUMMARY_PATH.write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
display(pd.Series(comparison, name="submission comparison"))

submission_experiment_id           BASEMLP-20260719-FEAT-AGE-ANOMALY-NB-03-SUB-01
feature_experiment_id                     BASEMLP-20260719-FEAT-AGE-ANOMALY-NB-03
feature_submission_path         submissions/submission_BASEMLP-20260719-FEAT-A...
baseline_submission_path        submissions/submission_BASEMLP-20260719-2H-01.csv
columns_equal                                                                True
id_exact_equal                                                               True
saleprice_exact_equal                                                       False
different_prediction_rows                                                    1459
max_abs_saleprice_difference                                         237758.88604
submission_checks               {'columns': True, 'rows': True, 'id_order': Tr...
kaggle_public_score                                                    unverified
kaggle_private_score                                                   unverified
Name: submission

## Takeaways

- 원본 연도를 보존한 self-contained notebook reference가 정상 실행됐다.
- 음수 연식은 결측 처리되고 `FutureYearAnomaly`는 train에서 1행에 적용됐다.
- feature 실험의 CV 변화는 +0.001820, OOF 변화는 +0.001568로 평균 성능이 악화되어 **reject**했다.
- 5개 fold 중 3개가 개선됐고 차이는 fold 변동성보다 작으므로, 이 결과를 개별 연식 피처의 독립 효과로 해석하지 않는다.
- 외부 프로젝트 스크립트 의존은 0건이며 Kaggle public/private 점수는 모두 **unverified**다.